# MethylGPT: Embedding Analysis

This notebook demonstrates how to analyze MethylGPT embeddings:

- **UMAP visualization** colored by metadata (tissue type, age, sex, disease, batch)
- **Clustering analysis** using K-Means
- **Silhouette score evaluation** to assess embedding quality
- **CpG-level embedding** visualization

**Prerequisites:** Run the `get_embeddings` tutorial first to generate an embeddings file, or this notebook will extract embeddings from scratch.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q methylgpt[tutorials]
    !pip install -q gdown
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
import os
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import umap

from methylgpt import MethylGPTModel, MethylVocab, create_dataloader
from methylgpt.inference import extract_embeddings

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load or Extract Embeddings

In [ ]:
# === OPTION A: Load pre-computed embeddings ===
EMBEDDINGS_FILE = "embeddings.pkl"  # From get_embeddings tutorial

# === OPTION B: Extract fresh embeddings ===
MODEL_DIR = Path("pretrained_models/methylgpt-medium")
CPG_LIST_FILE = "data/probe_ids_type3.csv"
PARQUET_DIR = "data/processed_type3_parquet_shuffled"
METADATA_FILE = "data/compiled_metadata.csv"

if Path(EMBEDDINGS_FILE).exists():
    with open(EMBEDDINGS_FILE, "rb") as f:
        data = pickle.load(f)
    embeddings = data["cell_emb"]
    sample_ids = data["cell_list"]
    print(f"Loaded embeddings: {embeddings.shape}")
else:
    # Extract embeddings
    with open(MODEL_DIR / "args.json", "r") as f:
        config = json.load(f)

    model_files = list(MODEL_DIR.glob("*.pt"))
    assert model_files, f"No .pt in {MODEL_DIR}"
    config["load_model"] = True
    config["pretrained_file"] = str(model_files[0])
    config["mask_ratio"] = 0

    vocab = MethylVocab(
        probe_id_dir=CPG_LIST_FILE, pad_token="<pad>",
        special_tokens=["<pad>", "<cls>", "<eoc>"], save_dir=None,
    )
    model = MethylGPTModel.from_pretrained(config, vocab)
    model.eval().to(device)
    if device.type == "cuda":
        model.half()

    parquet_files = sorted([
        os.path.join(PARQUET_DIR, f) for f in os.listdir(PARQUET_DIR)
        if f.endswith(".parquet")
    ])
    data_loader = create_dataloader([parquet_files[0]], batch_size=32)
    embeddings, sample_ids = extract_embeddings(model, data_loader, device=str(device), max_batches=200)
    print(f"Extracted embeddings: {embeddings.shape}")

## 2. Compute UMAP

In [ ]:
reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, metric="cosine", random_state=42)
umap_coords = reducer.fit_transform(embeddings)
print(f"UMAP computed: {umap_coords.shape}")

## 3. Load Metadata & Visualize

In [ ]:
# Create embedding DataFrame
emb_df = pd.DataFrame({
    "sample_id": sample_ids,
    "umap_1": umap_coords[:, 0],
    "umap_2": umap_coords[:, 1],
})

# Try to load metadata
if Path(METADATA_FILE).exists():
    meta = pd.read_csv(METADATA_FILE, low_memory=False)
    id_col = "gsm_id" if "gsm_id" in meta.columns else meta.columns[0]
    emb_df = emb_df.merge(meta, left_on="sample_id", right_on=id_col, how="left")
    print(f"Merged metadata: {len(emb_df)} samples")
    print(f"Available columns: {list(meta.columns[:15])}...")
else:
    print(f"No metadata file at {METADATA_FILE}. Plotting without annotations.")

In [ ]:
from aquarel import load_theme

theme = (
    load_theme("scientific")
    .set_grid(draw=False)
    .set_font(size=15)
    .set_ticks(direction="out")
    .set_axis_labels(pad=10)
)


def plot_umap_by_category(df, color_col, title, ax, max_categories=15):
    """Plot UMAP colored by categorical variable."""
    valid = df.dropna(subset=[color_col])
    if len(valid) == 0:
        ax.text(0.5, 0.5, f"No data for {color_col}", ha="center", va="center", transform=ax.transAxes)
        return

    # Keep top categories
    top_cats = valid[color_col].value_counts().head(max_categories).index
    valid = valid[valid[color_col].isin(top_cats)]

    categories = valid[color_col].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(categories)))

    for cat, color in zip(categories, colors):
        mask = valid[color_col] == cat
        n = mask.sum()
        label = f"{cat} ({n})" if n < 1000 else f"{cat} ({n//1000}k)"
        # Outline layer
        ax.scatter(valid.loc[mask, "umap_1"], valid.loc[mask, "umap_2"],
                   s=5, alpha=1, c="black", zorder=1)
        # Data layer
        ax.scatter(valid.loc[mask, "umap_1"], valid.loc[mask, "umap_2"],
                   s=3, alpha=0.5, c=[color], label=label, zorder=2)

    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, markerscale=3, frameon=False)


def plot_umap_by_numerical(df, color_col, title, ax, cmap="viridis"):
    """Plot UMAP colored by numerical variable."""
    valid = df.dropna(subset=[color_col])
    if len(valid) == 0:
        ax.text(0.5, 0.5, f"No data for {color_col}", ha="center", va="center", transform=ax.transAxes)
        return

    # Outline layer
    ax.scatter(valid["umap_1"], valid["umap_2"], s=5, c="black", alpha=1, zorder=1)
    # Data layer
    sc = ax.scatter(valid["umap_1"], valid["umap_2"],
                    s=3, alpha=0.5, c=valid[color_col], cmap=cmap, zorder=2)
    plt.colorbar(sc, ax=ax, label=color_col)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title(title)


# Plot UMAP by available metadata
categorical_cols = ["tissue", "sex", "disease"]
numerical_cols = ["age"]

available_cat = [c for c in categorical_cols if c in emb_df.columns and emb_df[c].notna().sum() > 10]
available_num = [c for c in numerical_cols if c in emb_df.columns and emb_df[c].notna().sum() > 10]

n_plots = len(available_cat) + len(available_num)

theme.apply()

if n_plots == 0:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1], s=5, c="black", alpha=1, zorder=1)
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1], s=3, alpha=0.5, c="steelblue", zorder=2)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title("MethylGPT Embeddings (no metadata available)")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    idx = 0
    for col in available_cat:
        plot_umap_by_category(emb_df, col, f"UMAP by {col}", axes[idx])
        idx += 1
    for col in available_num:
        plot_umap_by_numerical(emb_df, col, f"UMAP by {col}", axes[idx])
        idx += 1

theme.apply_transforms()

plt.savefig("embedding_umap_analysis.pdf", bbox_inches="tight")
plt.savefig("embedding_umap_analysis.png", dpi=600, bbox_inches="tight")
plt.show()

## 4. Clustering Analysis

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Try different numbers of clusters
k_range = range(2, 11)
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels, sample_size=min(5000, len(embeddings)))
    silhouette_scores.append(score)
    print(f"k={k}: Silhouette Score = {score:.4f}")

# Plot
theme.apply()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), silhouette_scores, "o-", linewidth=2, markersize=8, color="steelblue")
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Clustering Quality by Number of Clusters")
best_k = list(k_range)[np.argmax(silhouette_scores)]
ax.axvline(best_k, color="red", linestyle="--", alpha=0.7, label=f"Best k={best_k}")
ax.legend(frameon=False)

theme.apply_transforms()

plt.savefig("silhouette_analysis.pdf", bbox_inches="tight")
plt.savefig("silhouette_analysis.png", dpi=600, bbox_inches="tight")
plt.show()
print(f"\nBest k={best_k} with silhouette={max(silhouette_scores):.4f}")

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

theme.apply()

fig, ax = plt.subplots(figsize=(8, 6))
for i in range(best_k):
    mask = cluster_labels == i
    # Outline layer
    ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
               s=8, c="black", alpha=1, zorder=1)
    # Data layer
    ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
               s=5, alpha=0.5, label=f"Cluster {i} ({mask.sum()})", zorder=2)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title(f"K-Means Clustering (k={best_k})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

theme.apply_transforms()

plt.savefig("embedding_clusters.pdf", bbox_inches="tight")
plt.savefig("embedding_clusters.png", dpi=600, bbox_inches="tight")
plt.show()

## 5. CpG Embedding Analysis

In [ ]:
from methylgpt.inference import extract_cpg_embeddings

# Re-load model if needed (was freed from memory)
try:
    cpg_embs = extract_cpg_embeddings(model)
except:
    print("Model not in memory. Skipping CpG analysis.")
    cpg_embs = None

if cpg_embs is not None:
    # Skip special tokens (first 3)
    cpg_only = cpg_embs[3:]
    print(f"CpG embeddings: {cpg_only.shape}")

    # UMAP of CpG embeddings (subsample for speed)
    n_subsample = min(5000, len(cpg_only))
    indices = np.random.RandomState(42).choice(len(cpg_only), n_subsample, replace=False)
    cpg_subset = cpg_only[indices]

    reducer_cpg = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    cpg_umap = reducer_cpg.fit_transform(cpg_subset)

    theme.apply()

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(cpg_umap[:, 0], cpg_umap[:, 1], s=4, c="black", alpha=1, zorder=1)
    ax.scatter(cpg_umap[:, 0], cpg_umap[:, 1], s=2, alpha=0.3, c="steelblue", zorder=2)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title(f"CpG Embeddings ({n_subsample:,} sites)")

    theme.apply_transforms()

    plt.savefig("cpg_embedding_umap.pdf", bbox_inches="tight")
    plt.savefig("cpg_embedding_umap.png", dpi=600, bbox_inches="tight")
    plt.show()

## Next Steps

- **[Imputation Tutorial](../imputation/)** - Reconstruct missing methylation values
- **[Finetuning Tutorial](../finetuning_age_prediction/)** - Fine-tune MethylGPT for age prediction
- **[CpG Selection Tutorial](../cpg_selection/)** - Identify informative CpG sites
- **[Disease Prediction Tutorial](../disease_prediction/)** - Downstream survival analysis